In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style = "darkgrid")
from statsmodels.graphics.tsaplots import plot_acf

from src.database.connection import get_connection

import warnings
warnings.filterwarnings(
    "ignore",
    message = "pandas only supports SQLAlchemy connectable.*",
    category = UserWarning
)

In [ ]:
conn = get_connection()

query = """
    SELECT *
    FROM generation_eda
    ORDER BY start_time
"""

generation = pd.read_sql(query, conn)
generation["publish_time"] = pd.to_datetime(generation["publish_time"], utc = True)
generation["start_time"] = pd.to_datetime(generation["start_time"], utc = True)

query = """
    SELECT 
        *
    FROM demand_eda;
"""

demand = pd.read_sql(query, conn)
demand["start_time"] = pd.to_datetime(demand["start_time"], utc = True)

conn.close()

generation.head()

In [ ]:
generation.describe()

In [ ]:
generation.groupby("fuel_type")["start_time"].diff().value_counts().sort_index()

In [ ]:
generation.groupby("fuel_type").agg(rows = ("generation_mw", "size"))

In [ ]:
diffs = generation.groupby("fuel_type")["start_time"].diff()
generation.loc[diffs == pd.Timedelta("1 hour")]

In [ ]:
generation.head()

In [ ]:
fill_generation = generation.copy()

fill_generation = generation.sort_values(["fuel_type", "start_time"])
fill_generation["previous_generation"] = fill_generation.groupby("fuel_type")["generation_mw"].shift(1)
fill_generation["next_generation"] = fill_generation.groupby("fuel_type")["generation_mw"].shift(-1)
fill_generation["previous_start_time"] = fill_generation.groupby("fuel_type")["start_time"].shift(1)
fill_generation["interval"] = fill_generation["start_time"] - fill_generation["previous_start_time"]

wrong_intervals = fill_generation[fill_generation["interval"] == pd.Timedelta(hours = 1)].copy()
wrong_intervals["publish_time"] = wrong_intervals["start_time"]
wrong_intervals["start_time"] = wrong_intervals["start_time"] - pd.Timedelta(minutes = 30)
wrong_intervals["generation_mw"] = ((
    wrong_intervals["previous_generation"] + wrong_intervals["next_generation"]
    ) / 2).round(0)

wrong_intervals = wrong_intervals[["publish_time", "start_time", "fuel_type", "generation_mw"]]

generation = pd.concat([generation, wrong_intervals], ignore_index = True).sort_values("start_time")

In [ ]:
fill_demand = demand.copy()

fill_demand["previous_demand"] = fill_demand["true_demand_mw"].shift(1)
fill_demand["next_demand"] = fill_demand["true_demand_mw"].shift(-1)
fill_demand["previous_start_time"] = fill_demand["start_time"].shift(1)
fill_demand["interval"] = fill_demand["start_time"] - fill_demand["previous_start_time"]

wrong_intervals = fill_demand[fill_demand["interval"] == pd.Timedelta(hours = 1)].copy()
wrong_intervals["start_time"] = wrong_intervals["start_time"] - pd.Timedelta(minutes = 30)
wrong_intervals["true_demand_mw"] = ((
    wrong_intervals["previous_demand"] + wrong_intervals["next_demand"]
    ) / 2).round(0)

wrong_intervals = wrong_intervals[["start_time", "true_demand_mw"]]

demand = pd.concat([demand, wrong_intervals], ignore_index = True).sort_values("start_time")
demand = demand.rename(columns = {"start_time": "prediction_time"})

In [ ]:
generation_pivot = generation.pivot_table(
    index = "publish_time",
    columns = "fuel_type",
    values = "generation_mw",
    aggfunc = "last"
).reset_index().sort_values("publish_time")

generation_pivot.head()

In [ ]:
modelling = pd.merge_asof(
    left = demand,
    right = generation_pivot,
    left_on = "prediction_time",
    right_on = "publish_time",
    direction = "backward"
)

modelling.head()

In [ ]:
modelling["interval"] = modelling["prediction_time"] - modelling["publish_time"]
modelling["interval"].value_counts()

In [ ]:
wrong_intervals = modelling[modelling["interval"] > pd.Timedelta(0)]
wrong_intervals[["prediction_time", "publish_time", "interval"]].head(20)